# K-Head Discovery (BASE) — knowledge-task head discovery on base models

Reproduces the **base-model side of Appendix M (Table 20)**: independent discovery of
knowledge-task ("K") heads on the four base models, overlap with the published binding
heads (Jaccard + near-miss diagnostics), and the K-only causal knockout with the
convergence decision rule (cell 9). Shared helpers are imported from `common/` instead
of being redefined inline.

**Base-model port of `khead_discovery_instruct`.** Three structural changes vs the instruct version:
1. **Prompt format**: Wang base format (`format_for_base` = question + MC_WANG + ANSWER_START), no chat template. MC is not embedded in the raw texts (unlike the instruct notebook): `format_for_base` adds it exactly once.
2. **Option scoring**: conditional '(' scoring (`discover_after_paren_ids` + `COND_AFTER_IDS`), not `find_option_token_ids`. Nemo embeds '(' in ANSWER_START (`in_prompt`).
3. **Config**: base model paths; **the Gemma key is `gemma`, NOT `gemma2`** (base pipeline convention). GATE 4 targets are the published BASE |dK| values.

**Decision: K-ONLY.** No S-task extraction/CV, no S-side knockout: base |dS| (0.04–0.19) makes S percentages unusable. Cells 5/6/8/8b run on the knowledge task only.

Layout: 1 setup → 2 config+load → 2b base overrides → 3 K prompts + conditional discovery → 4 baseline K (GATE vs published base) → 5 attention extraction (K) → 6 discovery CV → 7 overlap vs published S-heads → 8 group KO (K-only) → 8b per-head KO (K-only) → 9 summary.


In [ ]:
MODEL_KEY = "mistral"  # one of {"mistral", "llama", "gemma", "nemo"}  (BASE key: gemma, NOT gemma2!)

# Per-model HEADS_TO_TEST for the cell-8b per-head knockout (cell-6 stable
# set + published binding heads).
PER_MODEL = {
    "mistral": {
        "HEADS_TO_TEST": [
            (7, 29), (8, 7), (8, 16), (8, 18), (9, 8), (9, 23),
            (11, 7), (11, 9), (11, 26), (12, 15), (12, 20), (12, 22),
            (12, 9), (13, 9), (14, 1), (14, 13), (14, 27), (15, 30), (16, 1),
        ],
    },
    "llama": {
        "HEADS_TO_TEST": [
            (7, 6), (7, 7), (8, 16), (8, 17), (9, 2), (9, 23),
            (11, 9), (11, 12), (11, 14), (12, 6), (13, 12), (13, 13),
            (13, 18), (14, 7), (14, 16), (22, 9),
        ],
    },
    "gemma": {
        "HEADS_TO_TEST": [
            (9, 2), (11, 14), (12, 14), (13, 13), (15, 2), (15, 11),
            (16, 10), (16, 14), (16, 15), (17, 9), (19, 3), (19, 7),
            (20, 2), (20, 6), (20, 7), (21, 0), (22, 5), (26, 2),
            (39, 6),
        ],
    },
    "nemo": {
        "HEADS_TO_TEST": [
            (5, 3), (5, 10), (7, 0), (8, 9), (8, 20), (8, 22),
            (10, 24), (11, 3), (12, 14), (12, 15), (15, 28), (16, 15),
            (16, 21), (16, 28), (17, 17), (17, 18), (18, 20),
        ],
    },
}


## Cell 2 — Config, shared helpers, model load

Model table, seed and output paths come from `common.config` (base table; **the Gemma key is `gemma`, NOT `gemma2`**). The shared text/data/prompt/knockout helpers are imported from `common/`.


In [ ]:
import sys, os
# Locate the repo root (the directory containing common/), whatever the kernel cwd
_p = os.path.abspath(".")
REPO_ROOT = _p if os.path.isdir(os.path.join(_p, "common")) else os.path.abspath("..")
assert os.path.isdir(os.path.join(REPO_ROOT, "common")), (
    "Cannot locate the repo root: run this notebook from its own directory or the repo root")
sys.path.insert(0, REPO_ROOT)
import time

from common import config

CFG = config.init(MODEL_KEY, "base")
SEED = config.SEED

from common.text_parsers import (norm_identity, extract_options,
                                 extract_scenario, parse_meta, replace_options)
from common.base.data import (load_n4, build_factorial_as_conditions,
                              build_knowledge_probes, KNOWLEDGE_OPT_C)
from common.base.prompts import (find_option_token_ids, format_for_base,
                                 discover_after_paren_ids,
                                 discover_after_paren_ids_in_prompt,
                                 find_token_spans, extract_options_text,
                                 find_identity_and_item_positions)
# edge_knockout is deliberately not imported from common.base.hooks: this
# notebook defines its own local copy in the machinery cell below
from common.stats_utils import ttest_clustered

# Local aliases used by the experiment cells below
ACTIVE_MODEL = config.ACTIVE_MODEL
HF_TOKEN = config.HF_TOKEN
DEVICE = config.DEVICE
DATA_DIR = config.DATA_DIR
OUTPUT_DIR = config.OUTPUT_DIR
MC_WANG = config.MC_WANG
MC = config.MC  # alias kept from the source config cell (MC = MC_WANG)
ANSWER_START = config.ANSWER_START

# This notebook keeps the instruct-style CFG key names ('model_path', 'label',
# 'has_softcapping') and aliases them onto the base config table (same data;
# the base table calls the softcapping flag 'softcap').
CFG = dict(CFG)
CFG["model_path"] = CFG["path"]
CFG["has_softcapping"] = CFG["softcap"]
CFG["label"] = {
    "mistral": "Mistral-7B-v0.3 (base)",
    "llama":   "Llama-3.1-8B (base)",
    "gemma":   "Gemma-2-9B (base)",
    "nemo":    "Mistral-Nemo-12B (base)",
}[MODEL_KEY]

print(f"OUTPUT_DIR = {OUTPUT_DIR}")
print(f"Active model: {CFG['label']}")
print(f"  binding heads (instruct-discovered): {CFG['heads']}")
print(f"  cond_approach: {CFG['cond_approach']} (token {repr(CFG['cond_token_str'])})")
print(f"  ANSWER_START ends with: ...{repr(ANSWER_START[-10:])}")


In [ ]:
import os
import re
import gc
import ast
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import importlib
import matplotlib.pyplot as plt

from pathlib import Path
from collections import defaultdict, Counter
from contextlib import contextmanager
from itertools import combinations

from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.utils import resample
from scipy import stats
from scipy.stats import ttest_rel

import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)


## Cell 2b — BASE overrides

`format_for_base`, offset-based span detection (`find_token_spans` / `find_identity_and_item_positions`) and conditional '(' scoring (`discover_after_paren_ids[_in_prompt]`) are imported from `common.base.prompts`; `edge_knockout` from `common.base.hooks`. The cell below defines the khead-specific pieces locally: `reset_eager_attention` (leak fix), `format_for_extraction`, `detect_spans_base`, `validate_spans` and `compute_K_scores_base`.


In [ ]:
# ================================================================
# CELL 2b — BASE-MODEL OVERRIDES
# format_for_base replaces format_for_chat; conditional '(' scoring
# replaces find_option_token_ids-based scoring; find_token_spans-based
# (offset) span detection replaces detect_spans for KO positions.
# ================================================================
from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS as _AAF


def reset_eager_attention():
    """Purge the wrapper edge_knockout leaks into ALL_ATTENTION_FUNCTIONS
    and clear the stale edge registry (leak: orig_entry is None on first
    entry, so the restore is skipped and forwards outside a KO context
    after the first KO are phantom-masked)."""
    global _EDGE_REGISTRY
    if hasattr(_AAF, '_local_mapping'):
        _AAF._local_mapping.pop('eager', None)
    _EDGE_REGISTRY = {}


# format_for_base / find_token_spans / extract_options_text /
# find_identity_and_item_positions / discover_after_paren_ids[_in_prompt]
# are imported from common.base.prompts. edge_knockout is deliberately not
# imported: it is defined locally in the machinery cell below.

def format_for_extraction(texts):
    """Attention-extraction format: question + MC, WITHOUT the answer
    preamble — the closest base analog of the instruct Stage-2 convention
    (chat-formatted with MC embedded but no 'Answer:' suffix)."""
    return [t + "\n\n" + MC_WANG for t in texts]


def detect_spans_base(formatted_text, oa, ob, item):
    """Adapter for the extraction loop: same output keys as detect_spans
    ({'item','opt_a','opt_b'}) but built on find_token_spans (offset-based,
    the base-pipeline convention)."""
    a = find_token_spans(tokenizer, formatted_text, oa)
    b = find_token_spans(tokenizer, formatted_text, ob)
    it = find_token_spans(tokenizer, formatted_text, item)
    if not it:
        for prefix in ["the ", "a ", "an ", ""]:
            it = find_token_spans(tokenizer, formatted_text, prefix + item)
            if it:
                break
    if not a or not b or not it:
        return None
    return {"item": it, "opt_a": a, "opt_b": b}


# validate_spans: span printout used by the cell-5 extraction loop.
def validate_spans(input_ids, spans, tokenizer, label=""):
    ids_list = input_ids.tolist() if torch.is_tensor(input_ids) else input_ids
    print(f"\n  [{label}]")
    for name, indices in spans.items():
        if indices is None:
            print(f"    {name}: NOT FOUND"); continue
        decoded = tokenizer.decode([ids_list[i] for i in indices])
        print(f"    {name}: '{decoded}' (tokens {indices[0]}-{indices[-1]})")


def compute_K_scores_base(texts, positions, heads, edge_mode, progress_every=200,
                          label=""):
    """K-scores with conditional '(' scoring + optional edge KO.
    Same math and same KO wiring as common.base.scoring._compute_scores:
    source = item tokens, target = identity tokens)."""
    scores = np.zeros(len(texts), dtype=np.float64)
    use_prefix = (CFG["cond_approach"] == "prefix")
    for i, text in enumerate(texts):
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        input_ids = enc["input_ids"].to(first_device)
        attn_mask = enc["attention_mask"].to(first_device)
        if use_prefix:
            input_ids = torch.cat([input_ids,
                torch.tensor([[COND_PREFIX_TID]], device=first_device)], dim=1)
            attn_mask = torch.cat([attn_mask,
                torch.ones(1, 1, device=first_device, dtype=attn_mask.dtype)], dim=1)
        pos = positions[i] if positions is not None else None
        if pos is None or not heads or edge_mode == 'none':
            with torch.no_grad():
                out = model(input_ids=input_ids, attention_mask=attn_mask)
        else:
            src = pos["item_tokens"]
            tgt = pos["B_tokens"] if edge_mode == "B_to_item" else pos["A_tokens"]
            with torch.no_grad(), edge_knockout(model, heads, src, tgt):
                out = model(input_ids=input_ids, attention_mask=attn_mask)
        lp = F.log_softmax(out.logits[0, -1, :].float(), dim=-1)
        lps = {}
        for opt in ["a", "b", "c"]:
            tids = COND_AFTER_IDS[opt]
            lps[opt] = (torch.logsumexp(lp[torch.tensor(tids, device=first_device)], 0).item()
                        if tids else float("-inf"))
        scores[i] = lps["c"] - np.logaddexp(lps["a"], lps["b"])
        del enc, input_ids, attn_mask, out, lp
        if progress_every and i % progress_every == 0 and i > 0:
            print(f"    {label}: {i}/{len(texts)}")
            torch.cuda.empty_cache()
    return scores

print("Base overrides ready (format_for_base, conditional scoring, offset spans,")
print("reset_eager_attention).")


## Edge-knockout machinery (deliberately notebook-local)

This notebook deliberately keeps its own copy of the knockout/reset machinery (leak-window semantics differ from common/*/hooks.py). The pipeline version in `common.base.hooks` clears `_EDGE_REGISTRY` in its `finally` block, whereas the version below leaves its wrapper in `ALL_ATTENTION_FUNCTIONS._local_mapping` on exit and never clears `_EDGE_REGISTRY`: `reset_eager_attention` (cell 2b above) purges both **in this notebook's namespace**. `compute_K_scores_base` (cell 2b) resolves `edge_knockout` in this notebook's namespace at call time, so it uses the local copy below.

In [ ]:
# ================================================================
# EDGE-KNOCKOUT MACHINERY — deliberately notebook-local
# (not imported from common/base/hooks.py, whose pipeline version clears
# _EDGE_REGISTRY in its finally block: different leak-window semantics).
# The published Table 20 numbers were produced with this version:
# edge_knockout leaves its wrapper in ALL_ATTENTION_FUNCTIONS._local_mapping
# on exit and never clears _EDGE_REGISTRY; reset_eager_attention (cell 2b)
# purges both in this notebook's namespace. compute_K_scores_base (cell 2b)
# resolves edge_knockout in this notebook's namespace at call time, so it
# uses this local copy.
# ================================================================
# ================================================================
# UNIFIED EDGE KNOCKOUT (supports soft-capping for Gemma-2)
# ================================================================

_EDGE_REGISTRY = {}

@contextmanager
def edge_knockout(model, heads, source_tokens, target_tokens):
    """Zero attention edges from source → target for specific heads.
    Automatically handles Gemma-2 soft-capping via CFG['has_softcapping'].
    """
    global _EDGE_REGISTRY
    if not heads or not source_tokens or not target_tokens:
        yield; return

    attn_module = importlib.import_module(CFG["attn_module"])
    from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS

    _EDGE_REGISTRY = {
        'heads': {k: set(v) for k, v in heads.items()},
        'source_tokens': sorted(set(source_tokens)),
        'target_tokens': sorted(set(target_tokens)),
    }

    original_eager = attn_module.eager_attention_forward
    n_heads = model.config.num_attention_heads
    n_kv_heads = getattr(model.config, 'num_key_value_heads', n_heads)
    kv_group_size = n_heads // n_kv_heads
    head_dim = getattr(model.config, "head_dim", None) or (
        model.config.hidden_size // n_heads)
    softcapping = getattr(model.config, 'attn_logit_softcapping', None) \
                  if CFG["has_softcapping"] else None

    def wrapped_eager(module, query, key, value, attention_mask, **kwargs):
        scaling = kwargs.pop('scaling', None)
        dropout = kwargs.pop('dropout', 0.0)
        layer_idx = getattr(module, 'layer_idx', None)
        heads_to_edit = _EDGE_REGISTRY['heads'].get(layer_idx, None)
        if not heads_to_edit:
            return original_eager(module, query, key, value, attention_mask,
                                  scaling=scaling, dropout=dropout, **kwargs)

        attn_output, attn_weights = original_eager(
            module, query, key, value, attention_mask,
            scaling=scaling, dropout=dropout, **kwargs)
        attn_output = attn_output.clone()

        # Detect head axis
        if query.shape[1] == n_heads:   qkv_head_axis = 1
        elif query.shape[2] == n_heads: qkv_head_axis = 2
        else: return attn_output, attn_weights

        if attn_output.shape[1] == n_heads:   out_head_axis = 1
        elif attn_output.shape[2] == n_heads: out_head_axis = 2
        else: return attn_output, attn_weights

        seq_len_q = query.shape[2] if qkv_head_axis == 1 else query.shape[1]
        seq_len_k = key.shape[2] if qkv_head_axis == 1 else key.shape[1]
        src = [s for s in _EDGE_REGISTRY['source_tokens'] if s < seq_len_k]
        tgt = [t for t in _EDGE_REGISTRY['target_tokens'] if t < seq_len_q]
        if not src or not tgt:
            return attn_output, attn_weights

        for h in heads_to_edit:
            kv_h = h // kv_group_size
            if qkv_head_axis == 1:
                q_h, k_h, v_h = query[:, h, :, :], key[:, kv_h, :, :], value[:, kv_h, :, :]
            else:
                q_h, k_h, v_h = query[:, :, h, :], key[:, :, kv_h, :], value[:, :, kv_h, :]

            attn_scores = torch.matmul(q_h, k_h.transpose(-2, -1)) * scaling
            # Gemma-2 soft-capping
            if softcapping is not None:
                attn_scores = attn_scores / softcapping
                attn_scores = torch.tanh(attn_scores)
                attn_scores = attn_scores * softcapping
            if attention_mask is not None and attention_mask.dim() == 4:
                if attention_mask.shape[1] == 1:
                    attn_scores = attn_scores + attention_mask[:, 0, :seq_len_q, :seq_len_k]
                else:
                    attn_scores = attn_scores + attention_mask[:, h, :seq_len_q, :seq_len_k]
            # Zero the target edges
            for t_idx in tgt:
                for s_idx in src:
                    attn_scores[:, t_idx, s_idx] = torch.finfo(attn_scores.dtype).min
            attn_probs = F.softmax(attn_scores, dim=-1)
            new_output = torch.matmul(attn_probs, v_h)
            if out_head_axis == 1:
                attn_output[:, h, :, :] = new_output
            else:
                attn_output[:, :, h, :] = new_output

        return attn_output, attn_weights

    # Monkey-patch
    attn_module.eager_attention_forward = wrapped_eager
    orig_entry = None
    try:
        orig_entry = ALL_ATTENTION_FUNCTIONS.get('eager', None)
        ALL_ATTENTION_FUNCTIONS['eager'] = wrapped_eager
    except: pass
    try:
        yield
    finally:
        attn_module.eager_attention_forward = original_eager
        if orig_entry is not None:
            try: ALL_ATTENTION_FUNCTIONS['eager'] = orig_entry
            except: pass


# ================================================================
# UNIFIED EDGE SCALE (α-scaling for dose-response + DiffAware)
# ================================================================

_SCALE_REGISTRY = {}

@contextmanager
def edge_scale(model, heads, source_tokens, target_tokens, alpha=0.0):
    """Scale attention edges: α=0 → full KO, α=1 → baseline, α>1 → amplify."""
    global _SCALE_REGISTRY
    if not heads or not source_tokens or not target_tokens or alpha == 1.0:
        yield; return

    attn_module = importlib.import_module(CFG["attn_module"])
    from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS

    _SCALE_REGISTRY = {
        'heads': {k: set(v) for k, v in heads.items()},
        'source_tokens': sorted(set(source_tokens)),
        'target_tokens': sorted(set(target_tokens)),
        'alpha': alpha,
    }

    original_eager = attn_module.eager_attention_forward
    n_heads = model.config.num_attention_heads
    n_kv_heads = getattr(model.config, 'num_key_value_heads', n_heads)
    kv_group_size = n_heads // n_kv_heads
    head_dim = getattr(model.config, "head_dim", None) or (
        model.config.hidden_size // n_heads)
    softcapping = getattr(model.config, 'attn_logit_softcapping', None) \
                  if CFG["has_softcapping"] else None

    def wrapped_eager(module, query, key, value, attention_mask, **kwargs):
        scaling = kwargs.pop('scaling', None)
        dropout = kwargs.pop('dropout', 0.0)
        layer_idx = getattr(module, 'layer_idx', None)
        heads_to_edit = _SCALE_REGISTRY['heads'].get(layer_idx, None)
        if not heads_to_edit:
            return original_eager(module, query, key, value, attention_mask,
                                  scaling=scaling, dropout=dropout, **kwargs)

        alpha_val = _SCALE_REGISTRY['alpha']

        # Normal forward
        attn_output, attn_weights = original_eager(
            module, query, key, value, attention_mask,
            scaling=scaling, dropout=dropout, **kwargs)
        attn_output = attn_output.clone()

        if query.shape[1] == n_heads:   qkv_head_axis = 1
        elif query.shape[2] == n_heads: qkv_head_axis = 2
        else: return attn_output, attn_weights

        if attn_output.shape[1] == n_heads:   out_head_axis = 1
        elif attn_output.shape[2] == n_heads: out_head_axis = 2
        else: return attn_output, attn_weights

        seq_len_q = query.shape[2] if qkv_head_axis == 1 else query.shape[1]
        seq_len_k = key.shape[2] if qkv_head_axis == 1 else key.shape[1]
        src = [s for s in _SCALE_REGISTRY['source_tokens'] if s < seq_len_k]
        tgt = [t for t in _SCALE_REGISTRY['target_tokens'] if t < seq_len_q]
        if not src or not tgt:
            return attn_output, attn_weights

        for h in heads_to_edit:
            kv_h = h // kv_group_size
            if qkv_head_axis == 1:
                q_h, k_h, v_h = query[:, h, :, :], key[:, kv_h, :, :], value[:, kv_h, :, :]
            else:
                q_h, k_h, v_h = query[:, :, h, :], key[:, :, kv_h, :], value[:, :, kv_h, :]

            attn_scores = torch.matmul(q_h, k_h.transpose(-2, -1)) * scaling
            if softcapping is not None:
                attn_scores = attn_scores / softcapping
                attn_scores = torch.tanh(attn_scores)
                attn_scores = attn_scores * softcapping
            if attention_mask is not None and attention_mask.dim() == 4:
                if attention_mask.shape[1] == 1:
                    attn_scores = attn_scores + attention_mask[:, 0, :seq_len_q, :seq_len_k]
                else:
                    attn_scores = attn_scores + attention_mask[:, h, :seq_len_q, :seq_len_k]

            # KO version: zero target edges
            attn_scores_ko = attn_scores.clone()
            for t_idx in tgt:
                for s_idx in src:
                    attn_scores_ko[:, t_idx, s_idx] = torch.finfo(attn_scores_ko.dtype).min
            attn_probs_ko = F.softmax(attn_scores_ko, dim=-1)
            out_ko = torch.matmul(attn_probs_ko, v_h)

            # Normal version
            attn_probs_normal = F.softmax(attn_scores, dim=-1)
            out_normal = torch.matmul(attn_probs_normal, v_h)

            # Blend: out = out_ko + α * (out_normal - out_ko)
            new_output = out_ko + alpha_val * (out_normal - out_ko)

            if out_head_axis == 1:
                attn_output[:, h, :, :] = new_output
            else:
                attn_output[:, :, h, :] = new_output

        return attn_output, attn_weights

    attn_module.eager_attention_forward = wrapped_eager
    orig_entry = None
    try:
        orig_entry = ALL_ATTENTION_FUNCTIONS.get('eager', None)
        ALL_ATTENTION_FUNCTIONS['eager'] = wrapped_eager
    except: pass
    try:
        yield
    finally:
        attn_module.eager_attention_forward = original_eager
        if orig_entry is not None:
            try: ALL_ATTENTION_FUNCTIONS['eager'] = orig_entry
            except: pass


# ================================================================
# UNIFIED LOGIT SCORING (with optional edge intervention)
# ================================================================

def compute_logit_scores_edge(model, tokenizer, formatted_texts, raw_texts,
                               heads, positions_list, edge_mode, option_tokens,
                               return_lps=False):
    """Compute S = logP(c) - logsumexp(logP(a), logP(b)) with optional edge KO.
    If return_lps=True, also returns a list of per-option logprob dicts."""
    scores = []
    all_lps = []
    first_device = next(model.parameters()).device
    for i, text in enumerate(formatted_texts):
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        enc = {k: v.to(first_device) for k, v in enc.items()}
        pos = positions_list[i]

        if not heads or pos is None:
            with torch.no_grad():
                outputs = model(**enc)
        else:
            if edge_mode == 'B_to_item':
                src, tgt = pos['item_tokens'], pos['B_tokens']
            elif edge_mode == 'A_to_item':
                src, tgt = pos['item_tokens'], pos['A_tokens']
            else:
                src, tgt = [], []
            with torch.no_grad(), edge_knockout(model, heads, src, tgt):
                outputs = model(**enc)

        logits = outputs.logits[0, -1, :].float()
        lp = F.log_softmax(logits, dim=-1)
        lp_a = torch.logsumexp(lp[option_tokens['a']], dim=0).item()
        lp_b = torch.logsumexp(lp[option_tokens['b']], dim=0).item()
        lp_c = torch.logsumexp(lp[option_tokens['c']], dim=0).item()
        scores.append(lp_c - torch.logsumexp(torch.tensor([lp_a, lp_b]), dim=0).item())
        if return_lps:
            all_lps.append({'a': lp_a, 'b': lp_b, 'c': lp_c})
        del enc, outputs, logits, lp
        if i % 100 == 0 and i > 0:
            torch.cuda.empty_cache()
    return (np.array(scores), all_lps) if return_lps else np.array(scores)


def compute_scores_scaled(model, tokenizer, formatted_texts, positions_list,
                           heads_dict, edge_mode, option_tokens, alpha=1.0):
    """Compute S-scores with edge_scale (α-scaling)."""
    scores = []
    first_device = next(model.parameters()).device
    for i, text in enumerate(formatted_texts):
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        enc = {k: v.to(first_device) for k, v in enc.items()}
        pos = positions_list[i]

        if alpha == 1.0 or pos is None or not heads_dict:
            with torch.no_grad():
                outputs = model(**enc)
        else:
            if edge_mode == 'B_to_item':
                src, tgt = pos['item_tokens'], pos['B_tokens']
            elif edge_mode == 'A_to_item':
                src, tgt = pos['item_tokens'], pos['A_tokens']
            else:
                src, tgt = [], []
            with torch.no_grad(), edge_scale(model, heads_dict, src, tgt, alpha=alpha):
                outputs = model(**enc)

        logits = outputs.logits[0, -1, :].float()
        lp = F.log_softmax(logits, dim=-1)
        lp_a = torch.logsumexp(lp[option_tokens['a']], dim=0).item()
        lp_b = torch.logsumexp(lp[option_tokens['b']], dim=0).item()
        lp_c = torch.logsumexp(lp[option_tokens['c']], dim=0).item()
        scores.append(lp_c - torch.logsumexp(torch.tensor([lp_a, lp_b]), dim=0).item())
        del enc, outputs, logits, lp
        if i % 100 == 0 and i > 0:
            torch.cuda.empty_cache()
    return np.array(scores)

In [ ]:
# ================================================================
# CELL 2 (load) — MODEL + TOKENIZER (bf16, device_map="auto", eager
# attention: the knockout machinery patches eager_attention_forward, and
# Gemma-2 soft-capping is handled inside edge_knockout via
# CFG["has_softcapping"]).
# ================================================================
_t_cell = time.time()

print(f"Loading {CFG['model_path']} ...")
tokenizer = AutoTokenizer.from_pretrained(
    CFG['model_path'], trust_remote_code=True, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    CFG['model_path'], dtype=torch.bfloat16, device_map="auto",
    trust_remote_code=True, token=HF_TOKEN,
    attn_implementation="eager",
)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

N_LAYERS = model.config.num_hidden_layers
N_HEADS = model.config.num_attention_heads
print(f"  {N_LAYERS} layers x {N_HEADS} heads")

option_tokens_raw = find_option_token_ids(tokenizer)
first_device = next(model.parameters()).device
option_tokens = {opt: torch.tensor(ids, device=first_device)
                 for opt, ids in option_tokens_raw.items()}
for opt, ids in option_tokens_raw.items():
    decoded = [tokenizer.decode([i]) for i in ids]
    print(f"  option '{opt}': {decoded}")

print("\n  NOTE (BASE): option_tokens above are reference-only; scoring uses")
print("  the conditional after_ids discovered in cell 3.")

print(f"\n[cell 2 load done in {time.time()-_t_cell:.1f}s]")

config.model, config.tokenizer, config.first_device = model, tokenizer, first_device


## Cell 3 — KNOWLEDGE-PROBE prompts (BASE format) + conditional-scoring discovery (+ GATES: 847 pairs, after_ids found, 3 decoded prompts)


In [ ]:
# ================================================================
# CELL 3 — KNOWLEDGE-PROBE PROMPTS (BASE FORMAT) + conditional-scoring
# discovery. The discovery sample is a binding prompt (B_cult[0]): the
# published after_ids were discovered on that prompt.
# ================================================================
_t_cell = time.time()

print("=" * 80)
print(f"CELL 3: KNOWLEDGE-PROBE PROMPTS — {CFG['label']}")
print("=" * 80)

cultural_items, neutral_items = load_n4(DATA_DIR)
data = build_factorial_as_conditions(cultural_items, seed=SEED)
n_total = len(data['B_cult'])
conditions = ['B_cult', 'B_unrel']

knowledge = build_knowledge_probes(data)
k_conditions = ['K_cult', 'K_unrel']

# GATE 3.1: pair counts
assert n_total == 847, f"GATE 3.1 FAILED: expected 847 factorial pairs, got {n_total}"
assert len(knowledge['K_cult']) == 847 and len(knowledge['K_unrel']) == 847, \
    "GATE 3.1 FAILED: knowledge probes != 847 in one of the conditions"
print(f"\n  [GATE 3.1 OK] 847 pairs in both K conditions, "
      f"{len(set(data['items_cult']))} unique items")

# GATE 3.1b: raw texts must not embed MC (the two # BASE edits in the
# helpers cell removed the instruct-notebook embedding)
assert MC_WANG not in data['B_cult'][0] and MC_WANG not in knowledge['K_cult'][0], \
    "GATE 3.1b FAILED: MC is embedded in the raw texts — revert to the base helpers " \
    "(format_for_base would add it a second time)"
print("  [GATE 3.1b OK] raw texts are MC-free; format_for_base adds MC_WANG once")

# BASE formatting (no chat template)
texts_fmt_k = {c: format_for_base(knowledge[c]) for c in k_conditions}
texts_fmt   = {c: format_for_base(data[c]) for c in conditions}  # discovery sample

# Conditional scoring discovery
COND_PREFIX_TID, COND_AFTER_IDS = None, None
_sample = texts_fmt['B_cult'][0]
if CFG["cond_approach"] == "prefix":
    COND_PREFIX_TID, COND_AFTER_IDS, _cov = discover_after_paren_ids(
        tokenizer, model, first_device, _sample, cond_token_str=CFG["cond_token_str"])
    print(f"\n  Conditional scoring: prefix {repr(CFG['cond_token_str'])} "
          f"= TID {COND_PREFIX_TID}")
else:
    COND_AFTER_IDS, _cov = discover_after_paren_ids_in_prompt(
        tokenizer, model, first_device, _sample)
    print(f"\n  Conditional scoring: '(' embedded in ANSWER_START (single forward)")
print(f"  after_ids: a={COND_AFTER_IDS['a']}, b={COND_AFTER_IDS['b']}, "
      f"c={COND_AFTER_IDS['c']}")
print(f"  Sample coverage P(a+b+c | '('): {_cov:.4f}")
assert all(COND_AFTER_IDS[o] for o in 'abc'), \
    "GATE 3.2 FAILED: missing after_ids for at least one option"
if _cov < 0.3:
    print("  !! WARNING: low option coverage — inspect the decoded prompts below "
          "before trusting K-scores")
else:
    print("  [GATE 3.2 OK] after_ids found for a/b/c")

# GATE 3.3: 3 fully decoded K prompts (exactly what the model sees)
print("\n  [GATE 3.3] decoded K prompts:")
for i in range(3):
    for cond in k_conditions:
        ids = tokenizer(texts_fmt_k[cond][i], return_tensors="pt")["input_ids"][0]
        print(f"\n  ----- pair {i} | {cond} | item = {data['items_cult'][i]} "
              f"| assoc_pos = {data['assoc_pos'][i]} -----")
        print("  | " + tokenizer.decode(ids).replace("\n", "\n  | "))
print("\n  Manually check: the prompt ends with ANSWER_START "
      "('...choose letter ' — or '...letter (' for Nemo); no chat-template tokens;"
      "\n  MC_WANG appears exactly ONCE.")

print(f"\n[cell 3 done in {time.time()-_t_cell:.1f}s]")


## Cell 4 — Baseline K-scores, conditional '(' scoring (+ GATE: |ΔK| must reproduce the published BASE value ±5%, HARD STOP)


In [ ]:
# ================================================================
# CELL 4 — BASELINE K-SCORES (conditional '(' scoring) on both
# conditions. GATE: |dK| must match the published BASE value within
# +/-5% or HARD STOP. The base |dK| are small (0.16-0.96), so the
# absolute slack is tight: a failure means a formatting / after_ids /
# model-key mismatch (gemma vs gemma2!), not noise.
# ================================================================
_t_cell = time.time()

PUBLISHED_ABS_DK_BASE = {"mistral": 0.262, "gemma": 0.955, "llama": 0.161, "nemo": 0.418}
DK_REL_TOL = 0.05  # +/-5%

K_BASELINE_CACHE = OUTPUT_DIR / f"{ACTIVE_MODEL}_base_K_baseline_scores.pkl"

if K_BASELINE_CACHE.exists():
    with open(K_BASELINE_CACHE, "rb") as f:
        results_kscore = pickle.load(f)
    print(f"  [cache hit] {K_BASELINE_CACHE.name} — skipping recomputation")
else:
    reset_eager_attention()
    results_kscore = {}
    for cond in k_conditions:
        K = compute_K_scores_base(texts_fmt_k[cond], None, {}, 'none',
                                  progress_every=200, label=f"baseline/{cond}")
        results_kscore[cond] = {'K': K}
    with open(K_BASELINE_CACHE, "wb") as f:
        pickle.dump(results_kscore, f)
    print(f"  Saved {K_BASELINE_CACHE}")

print(f"\n  {'':10s}  {'Mean K':>8s}  {'Med K':>8s}")
for cond in k_conditions:
    r = results_kscore[cond]['K']
    print(f"  {cond:10s}  {r.mean():8.3f}  {np.median(r):8.3f}")

delta_K = results_kscore['K_cult']['K'].mean() - results_kscore['K_unrel']['K'].mean()
target_dK = PUBLISHED_ABS_DK_BASE[ACTIVE_MODEL]
rel_dev = abs(abs(delta_K) - target_dK) / target_dK
print(f"\n  delta(K) = {delta_K:.4f}   |dK| = {abs(delta_K):.4f}")
print(f"  published |dK| ({ACTIVE_MODEL} base) = {target_dK}   "
      f"rel. deviation = {100*rel_dev:.2f}%")

if rel_dev > DK_REL_TOL:
    raise RuntimeError(
        f"GATE 4 FAILED: |dK| = {abs(delta_K):.4f} deviates {100*rel_dev:.1f}% from "
        f"the published BASE value {target_dK} (tolerance +/-5%). Check ACTIVE_MODEL "
        f"(the base Gemma key is 'gemma', NOT 'gemma2'), format_for_base, the "
        f"after_ids discovery, and the N4 file BEFORE running anything downstream.")
print(f"  [GATE 4 OK] |dK| reproduces the published base value within +/-5%")

print(f"\n[cell 4 done in {time.time()-_t_cell:.1f}s]")


## Cell 5 — Attention extraction on K-prompts, K-ONLY (the expensive cell; cached) (+ GATE: shape `[n_pairs, n_layers, n_heads, 3]`, NaN/span-failure < 2%)

Extraction prompts use `format_for_extraction` (question + MC, **without** ANSWER_START): the closest base analog of the instruct Stage-2 convention (MC present, answer cue absent). Spans via `detect_spans_base` (offset-based).


In [ ]:
# ================================================================
# ATTENTION EXTRACTION
# ================================================================

def extract_attention_scores(model, tokenizer, prompt_text, spans):
    """Extract binding scores for all (layer, head) pairs."""
    enc = tokenizer(prompt_text, return_tensors="pt")
    enc = {k: v.to(first_device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc, output_attentions=True, use_cache=False)
    n_layers = len(out.attentions)
    n_heads_model = out.attentions[0].shape[1]
    seq_len = out.attentions[0].shape[2]
    has_item = spans['item'] is not None
    item_idx = np.array(spans['item']) if has_item else None
    a_idx = np.array(spans['opt_a'])
    b_idx = np.array(spans['opt_b'])
    scores = {
        'bind_a_to_item': np.full((n_layers, n_heads_model), np.nan),
        'bind_b_to_item': np.full((n_layers, n_heads_model), np.nan),
        'bind_avg':       np.full((n_layers, n_heads_model), np.nan),
    }
    for l in range(n_layers):
        attn = out.attentions[l][0].float().cpu().numpy()
        if has_item:
            scores['bind_a_to_item'][l] = attn[:, a_idx][:, :, item_idx].sum(axis=2).mean(axis=1)
            scores['bind_b_to_item'][l] = attn[:, b_idx][:, :, item_idx].sum(axis=2).mean(axis=1)
            scores['bind_avg'][l] = (scores['bind_a_to_item'][l] +
                                     scores['bind_b_to_item'][l]) / 2.0
    del out, enc
    torch.cuda.empty_cache()
    return scores


# ================================================================
# CV UTILITIES
# ================================================================

N_OUTER_FOLDS = 5
N_INNER_FOLDS = 5
L1_Cs = [0.001, 0.01, 0.1, 1.0, 10.0]
TOP_K_HEADS = 10

def build_Xy(features_match, features_unrel, scenarios, feature_name):
    n = len(features_match)
    if n == 0: return None, None, None, None
    n_layers, n_heads_model = features_match[0][feature_name].shape
    X = np.zeros((2 * n, n_layers * n_heads_model))
    for i in range(n):
        X[i] = features_match[i][feature_name].flatten()
        X[n + i] = features_unrel[i][feature_name].flatten()
    if np.any(np.isnan(X)):
        nan_mask = np.isnan(X).any(axis=1)
        n_half = n
        valid_examples = ~(nan_mask[:n_half] | nan_mask[n_half:])
        n_keep = valid_examples.sum()
        if n_keep < 10: return None, None, None, None
        keep_idx = np.where(valid_examples)[0]
        keep_all = np.concatenate([keep_idx, keep_idx + n_half])
        X = X[keep_all]
        filtered_scenarios = [scenarios[i] for i in keep_idx]
        n = n_keep
    else:
        filtered_scenarios = list(scenarios)
    y = np.concatenate([np.ones(n), np.zeros(n)])
    groups = np.array(filtered_scenarios + filtered_scenarios)
    return X, y, groups, (n_layers, n_heads_model)


def cv_select_C(X, y, groups, n_splits=5, Cs=None):
    if Cs is None: Cs = L1_Cs
    actual_splits = min(n_splits, len(set(groups)))
    if actual_splits < 2: return 0.1, 0.5
    gkf = GroupKFold(n_splits=actual_splits)
    best_auc, best_C = -1, Cs[len(Cs) // 2]
    for C in Cs:
        pipe = Pipeline([("scaler", StandardScaler()),
            ("lr", LogisticRegression(penalty="l1", C=C, solver="liblinear",
                                      max_iter=1000, random_state=SEED))])
        try:
            aucs = cross_val_score(pipe, X, y, cv=gkf, groups=groups, scoring="roc_auc")
            if aucs.mean() > best_auc:
                best_auc, best_C = aucs.mean(), C
        except: pass
    return best_C, best_auc


def topk_heads(coef_2d, k=10):
    flat = np.abs(coef_2d).ravel()
    idx = np.argsort(flat)[::-1][:k]
    n_h = coef_2d.shape[1]
    return [(int(i // n_h), int(i % n_h)) for i in idx if coef_2d.ravel()[i] != 0]


def run_outer_cv(all_features, scenarios_valid, feature_name="bind_avg",
                 K=N_OUTER_FOLDS, k_heads=TOP_K_HEADS):
    X, y, groups, shape = build_Xy(
        all_features["B_cult"], all_features["B_unrel"],
        scenarios_valid, feature_name)
    if X is None:
        print(f"  ⚠ Cannot build X for {feature_name}"); return [], {}
    n_unique = len(set(groups))
    actual_K = min(K, n_unique)
    if actual_K < 2: return [], {}

    gkf = GroupKFold(n_splits=actual_K)
    fold_results = []
    print(f"\n  Outer {actual_K}-fold CV on {feature_name} "
          f"({X.shape[0]} rows, {n_unique} scenarios)")
    print(f"  {'Fold':<6s}  {'C':>6s}  {'innerAUC':>9s}  {'testAUC':>8s}  "
          f"{'#nz':>4s}  {'Top heads'}")
    print(f"  {'-' * 70}")

    for fold, (tr_idx, te_idx) in enumerate(gkf.split(X, y, groups=groups)):
        X_tr, y_tr, g_tr = X[tr_idx], y[tr_idx], groups[tr_idx]
        X_te, y_te = X[te_idx], y[te_idx]
        best_C, inner_auc = cv_select_C(
            X_tr, y_tr, g_tr, n_splits=min(N_INNER_FOLDS, len(set(g_tr))))
        pipe = Pipeline([("scaler", StandardScaler()),
            ("lr", LogisticRegression(penalty="l1", C=best_C, solver="liblinear",
                                      max_iter=1000, random_state=SEED))])
        pipe.fit(X_tr, y_tr)
        coef = pipe.named_steps["lr"].coef_[0].reshape(shape)
        n_nonzero = int((coef != 0).sum())
        try:
            test_auc = roc_auc_score(y_te, pipe.predict_proba(X_te)[:, 1])
        except: test_auc = 0.5
        heads_fold = topk_heads(coef, k=k_heads)
        heads_str = ", ".join(f"L{l:02d}H{h:02d}" for l, h in heads_fold[:5])
        print(f"  {fold:<6d}  {best_C:6.3f}  {inner_auc:9.3f}  {test_auc:8.3f}  "
              f"{n_nonzero:4d}  {heads_str}")
        fold_results.append({
            "fold": fold, "best_C": best_C, "inner_auc": inner_auc,
            "test_auc": test_auc, "n_nonzero": n_nonzero,
            "coef": coef, "heads": heads_fold,
            "train_idx": tr_idx, "test_idx": te_idx,
        })

    # Aggregate
    head_counter = Counter()
    for f in fold_results:
        for lh in f["heads"]:
            head_counter[tuple(lh)] += 1
    stable_heads = [(lh, cnt) for lh, cnt in head_counter.most_common()
                    if cnt >= max(2, len(fold_results) // 2)]
    mean_coef = np.mean([f["coef"] for f in fold_results], axis=0)

    summary = {
        "test_auc_mean": np.mean([f["test_auc"] for f in fold_results]),
        "test_auc_std":  np.std([f["test_auc"] for f in fold_results]),
        "stable_heads":  stable_heads,
        "head_counter":  dict(head_counter),
        "mean_coef":     mean_coef,
        "n_folds":       len(fold_results),
    }

    print(f"\n  Test AUC: {summary['test_auc_mean']:.3f} ± {summary['test_auc_std']:.3f}")
    if stable_heads:
        print(f"  Stable heads (≥ {len(fold_results)//2+1}/{len(fold_results)} folds):")
        for (l, h), cnt in stable_heads:
            sign = "↑ match" if mean_coef[l, h] > 0 else "↓ match"
            print(f"    L{l:02d}H{h:02d}: {cnt}/{len(fold_results)} folds, "
                  f"mean coef={mean_coef[l,h]:+.4f} ({sign})")

    return fold_results, summary


In [ ]:
# ================================================================
# CELL 5 (driver) — ATTENTION EXTRACTION (K-ONLY, BASE), cached to results/
# Base port of the instruct Stage-2 loop: prompts via format_for_extraction
# (question + MC, no ANSWER_START: the analog of the instruct 'no answer
# suffix' convention); spans via detect_spans_base (offset-based).
# S-task extraction intentionally dropped (K-only decision).
# ================================================================
_t_cell = time.time()

K_ATTN_CACHE = OUTPUT_DIR / f"{ACTIVE_MODEL}_base_K_attn_features.pkl"


def extract_task_features(task_label, task_texts_raw, task_conds, cache_path):
    """Stage-2 extraction loop on one task (base formatting)."""
    if cache_path.exists():
        with open(cache_path, "rb") as f:
            out = pickle.load(f)
        print(f"  [cache hit] {cache_path.name}: "
              f"{len(out['valid_indices'])} valid pairs, "
              f"{len(out['skipped_indices'])} skipped — skipping recomputation")
        return out

    all_features = {c: [] for c in task_conds}
    valid_indices, scenarios_valid, skipped_indices = [], [], []

    for i in range(n_total):
        item_cult = data['items_cult'][i]
        cond_info, prompts, spans_all = {}, {}, {}
        all_ok = True
        for cond in task_conds:
            full_text = task_texts_raw[cond][i]
            q_text = full_text.split("\n\n")[0]
            oa, ob = extract_options(q_text)
            cond_info[cond] = {'question': q_text, 'opt_a': oa, 'opt_b': ob}
            # BASE: extraction format = question + MC (no ANSWER_START)
            prompt = format_for_extraction([full_text])[0]
            sp = detect_spans_base(prompt, oa, ob, item_cult)
            if sp is None:
                all_ok = False
                break
            prompts[cond] = prompt
            spans_all[cond] = sp
        if not all_ok:
            skipped_indices.append(i)
            continue
        if len(valid_indices) < 3:  # span validation printout, first 3 pairs
            for cond in task_conds:
                enc = tokenizer(prompts[cond], return_tensors="pt")
                validate_spans(enc["input_ids"][0], spans_all[cond], tokenizer,
                               label=f"{task_label} {cond} pair {i}: "
                                     f"{cond_info[cond]['opt_a']} vs "
                                     f"{cond_info[cond]['opt_b']}")
                del enc
        for cond in task_conds:
            scores = extract_attention_scores(model, tokenizer, prompts[cond],
                                              spans_all[cond])
            all_features[cond].append(scores)
        valid_indices.append(i)
        scenarios_valid.append(data['scenarios'][i])
        if len(valid_indices) % 100 == 0:
            print(f"    {task_label}: {len(valid_indices)} valid / {i+1} seen "
                  f"(skipped {len(skipped_indices)})  [{time.time()-_t_cell:.0f}s]")

    out = {
        'features': all_features,
        'valid_indices': valid_indices,
        'scenarios_valid': scenarios_valid,
        'skipped_indices': skipped_indices,
        'meta': {'model': ACTIVE_MODEL, 'variant': 'base', 'task': task_label,
                 'n_total': n_total, 'seed': SEED,
                 'n_layers': N_LAYERS, 'n_heads': N_HEADS},
    }
    with open(cache_path, "wb") as f:
        pickle.dump(out, f)
    print(f"  Saved {cache_path}  ({os.path.getsize(cache_path)/1e6:.1f} MB)")
    return out


def stack_features(feat_list):
    """[n_pairs, n_layers, n_heads, 3] with feature order (f_a, f_b, f_avg)."""
    return np.stack([
        np.stack([f['bind_a_to_item'], f['bind_b_to_item'], f['bind_avg']], axis=-1)
        for f in feat_list])


def gate_features(attn, task_conds, label, hard=True):
    """GATE 5: shape + NaN + span-failure rate (< 2% of pairs)."""
    n_valid = len(attn['valid_indices'])
    n_skip = len(attn['skipped_indices'])
    worst_nan = 0
    for cond in task_conds:
        stk = stack_features(attn['features'][cond])
        expected = (n_valid, N_LAYERS, N_HEADS, 3)
        assert stk.shape == expected, \
            f"GATE 5 FAILED [{label}/{cond}]: feature shape {stk.shape} != {expected}"
        n_nan = int(np.isnan(stk).any(axis=(1, 2, 3)).sum())
        worst_nan = max(worst_nan, n_nan)
        print(f"  [{label}/{cond}] feature matrix {stk.shape}  NaN pairs: {n_nan}")
    fail_rate = (n_skip + worst_nan) / n_total
    print(f"  [{label}] span failures: {n_skip}/{n_total} ({100*n_skip/n_total:.2f}%)"
          f"   excluded pair indices: {attn['skipped_indices'] if n_skip else '[]'}")
    msg = (f"GATE 5 [{label}]: NaN/span-failure rate = {100*fail_rate:.2f}% "
           f"(must be < 2%)")
    if fail_rate < 0.02:
        print(f"  [GATE 5 OK] {msg}")
    elif hard:
        raise RuntimeError("GATE 5 FAILED: " + msg)
    else:
        print("  !! WARNING (soft gate): " + msg)


print("=" * 80)
print(f"CELL 5: ATTENTION EXTRACTION (K-ONLY, BASE) — {CFG['label']}")
print("=" * 80)

print("\n  K-task (knowledge probes):")
attn_K = extract_task_features("K-task", knowledge, k_conditions, K_ATTN_CACHE)
gate_features(attn_K, k_conditions, "K-task", hard=True)

attn_S = None
print("\n  K-ONLY decision: S-task extraction skipped (base |dS| too noisy; "
      "the vice-versa near-miss in cell 7 is unavailable by design).")

print(f"\n[cell 5 done in {time.time()-_t_cell:.1f}s]")


## Cell 6 — Head discovery: Stage-2 CV on the K task (+ GATE: mean AUC > 0.65; stable set = selected in ≥3/5 folds)


In [ ]:
# ================================================================
# CELL 6 — HEAD DISCOVERY (Stage-2 CV, run on the K features)
# L1 LogisticRegression (penalty='l1', solver='liblinear'), 5-fold
# GroupKFold grouped by cultural item, C from [0.001..10] by inner CV,
# ROC-AUC scoring. Stability: head selected in >= 3 of 5 folds.
# Also prints the union of the three stable sets (+ published binding
# heads if absent), ready to paste into HEADS_TO_TEST (cell 8b).
# ================================================================
_t_cell = time.time()

FEATURE_NAMES = ['bind_avg', 'bind_a_to_item', 'bind_b_to_item']
MIN_FOLDS_STABLE = 3
AUC_GATE = 0.65


def run_task_cv(attn, task_conds, task_label):
    """Alias the task's (match, mismatch) conditions onto the 'B_cult'/'B_unrel'
    keys that the Stage-2 code expects (match -> y=1, mismatch -> y=0)."""
    feats = {'B_cult': attn['features'][task_conds[0]],
             'B_unrel': attn['features'][task_conds[1]]}
    cv = {}
    for feat in FEATURE_NAMES:
        print(f"\n  ### {task_label} CV — feature '{feat}' ###")
        folds, summary = run_outer_cv(feats, attn['scenarios_valid'], feature_name=feat)
        cv[feat] = {'folds': folds, 'summary': summary}
    return cv


def stable_set(cv, feat='bind_avg', min_folds=MIN_FOLDS_STABLE):
    s = cv[feat]['summary']
    if not s:
        return []
    return sorted(tuple(lh) for lh, cnt in s.get('stable_heads', []) if cnt >= min_folds)


print("=" * 80)
print(f"K-TASK HEAD DISCOVERY — {CFG['label']}")
print("=" * 80)
cv_K = run_task_cv(attn_K, k_conditions, "K-task")

if attn_S is not None:
    print("\n" + "=" * 80)
    print(f"S-TASK CV (positive control + vice-versa near-miss) — {CFG['label']}")
    print("=" * 80)
    cv_S = run_task_cv(attn_S, conditions, "S-task")
else:
    cv_S = None

# Primary K-head set: bind_avg, >= 3/5 folds
K_HEADS_LIST = stable_set(cv_K, 'bind_avg')
K_HEADS_STRICT = stable_set(cv_K, 'bind_avg', min_folds=4)

# GATE 6: discovery quality
fold_aucs = [f['test_auc'] for f in cv_K['bind_avg']['folds']]
auc_K = float(np.mean(fold_aucs)) if fold_aucs else float('nan')
print("\n[GATE 6] K-task discovery quality (bind_avg):")
print(f"  per-fold test AUC: {['%.3f' % a for a in fold_aucs]}")
print(f"  mean test AUC = {auc_K:.3f}  (gate: > {AUC_GATE})")
K_DISCOVERY_WEAK = not (auc_K > AUC_GATE)
if K_DISCOVERY_WEAK:
    print("  " + "!" * 72)
    print("  !! WARNING: K-head discovery is WEAK on this model (mean AUC <= 0.65).")
    print("  !! The K-head set is unreliable; the overlap analysis (cells 7-9) should")
    print("  !! be treated as INCONCLUSIVE for this model.")
    print("  " + "!" * 72)
else:
    print("  [GATE 6 OK] discovery AUC is adequate")

print(f"\n  K-heads  (>= {MIN_FOLDS_STABLE}/5 folds, bind_avg): "
      f"{[f'L{l}H{h}' for l, h in K_HEADS_LIST]}")
print(f"  K-heads  (>= 4/5 folds, strict):        "
      f"{[f'L{l}H{h}' for l, h in K_HEADS_STRICT]}")
for feat in FEATURE_NAMES[1:]:
    alt = stable_set(cv_K, feat)
    print(f"  [context] {feat:>15s} stable set: {[f'L{l}H{h}' for l, h in alt]}")

# Union of the three stable sets + published binding heads if absent
union_set = set()
for feat in FEATURE_NAMES:
    union_set |= set(stable_set(cv_K, feat))
n_union = len(union_set)
pub_heads = sorted((l, h) for l, hs in CFG['heads'].items() for h in hs)
forced = [lh for lh in pub_heads if lh not in union_set]
HEADS_UNION = sorted(union_set | set(pub_heads))

print(f"\n  UNION of the 3 stable sets (>= {MIN_FOLDS_STABLE}/5 folds): "
      f"{n_union} heads")
print(f"    {[f'L{l}H{h}' for l, h in sorted(union_set)]}")
if forced:
    print(f"  + published binding heads NOT found by the K discovery "
          f"({len(forced)}): {[f'L{l}H{h}' for l, h in forced]}")
else:
    print("  + all published binding heads already in the union")
print(f"  TOTAL to test in cell 8b: {len(HEADS_UNION)} heads")
print("\n  HEADS_TO_TEST = [")
for j in range(0, len(HEADS_UNION), 6):
    print("      " + " ".join(f"({l}, {h})," for l, h in HEADS_UNION[j:j+6]))
print("  ]")

# Positive control: the S-CV should re-find the published S-heads
if cv_S is not None:
    S_FOUND = stable_set(cv_S, 'bind_avg')
    _missing = [lh for lh in pub_heads if lh not in S_FOUND]
    print(f"\n  [positive control] S-task stable set (bind_avg): "
          f"{[f'L{l}H{h}' for l, h in S_FOUND]}")
    if _missing:
        print(f"  !! WARNING: published S-heads not re-found by the S-CV: "
              f"{[f'L{l}H{h}' for l, h in _missing]}")
        print("  !! (published heads were finalized with the causal Stage-3 filter on top")
        print("  !!  of the CV, so treat this as a soft check — but do inspect the fold table)")
    else:
        print("  [OK] all published S-heads re-discovered on the binding task")

with open(OUTPUT_DIR / f"{ACTIVE_MODEL}_base_khead_cv.pkl", "wb") as f:
    pickle.dump({'cv_K': cv_K, 'cv_S': cv_S,
                 'K_HEADS_LIST': K_HEADS_LIST, 'K_HEADS_STRICT': K_HEADS_STRICT,
                 'HEADS_UNION': HEADS_UNION, 'union_K_only': sorted(union_set),
                 'forced_binding_heads': forced,
                 'auc_K': auc_K, 'fold_aucs': fold_aucs,
                 'K_DISCOVERY_WEAK': K_DISCOVERY_WEAK}, f)
print(f"\n  Saved CV results to {OUTPUT_DIR / (ACTIVE_MODEL + '_base_khead_cv.pkl')}")

print(f"\n[cell 6 done in {time.time()-_t_cell:.1f}s]")


## Cell 7 — Overlap analysis: K-heads vs published S-heads (Jaccard + near-miss + layer bands)


In [ ]:
# ================================================================
# CELL 7 — OVERLAP ANALYSIS
# Hardcoded published S-heads; intersection / Jaccard; near-miss
# diagnostic (fold counts + mean |coef| + coefficient rank) in both
# directions, so "different heads" can be distinguished from
# "same heads just below the stability threshold".
# ================================================================
_t_cell = time.time()

PUBLISHED_S_HEADS = {
    "mistral": [(8, 16), (9, 23), (12, 9)],
    "gemma2":  [(11, 14), (13, 13), (16, 15)],
    "gemma":   [(11, 14), (13, 13), (16, 15)],  # BASE pipeline key
    "llama":   [(7, 7), (8, 17)],
    "nemo":    [(8, 9), (10, 24)],
}
S_HEADS_LIST = sorted(PUBLISHED_S_HEADS[ACTIVE_MODEL])
_cfg_heads = sorted((l, h) for l, hs in CFG['heads'].items() for h in hs)
assert _cfg_heads == S_HEADS_LIST, \
    f"Consistency check failed: CFG heads {_cfg_heads} != published table {S_HEADS_LIST}"

K_set, S_set = set(K_HEADS_LIST), set(S_HEADS_LIST)
inter = sorted(K_set & S_set)
union = sorted(K_set | S_set)
jaccard = len(inter) / len(union) if union else float('nan')


def fmt_heads(hs):
    return "{" + ", ".join(f"L{l}H{h}" for l, h in sorted(hs)) + "}"


print("=" * 80)
print(f"OVERLAP ANALYSIS — {CFG['label']}")
print("=" * 80)
print(f"  K-heads (knowledge task, >= 3/5 folds): {fmt_heads(K_set)}")
print(f"  S-heads (published binding heads):      {fmt_heads(S_set)}")
print(f"  intersection: {fmt_heads(inter)}")
print(f"  Jaccard = {jaccard:.3f}   (|intersection| = {len(inter)}, |union| = {len(union)})")


def near_miss_table(cv, heads, cv_label):
    """For each head: how often selected across the 5 folds of this CV,
    its mean |coefficient|, and its rank by mean |coef| among all L*H heads."""
    out = {}
    if cv is None or not heads:
        print(f"\n  (near-miss table for {cv_label} unavailable)")
        return out
    s = cv['bind_avg']['summary']
    if not s:
        print(f"\n  (no CV summary for {cv_label})")
        return out
    counter = {tuple(k): v for k, v in s['head_counter'].items()}
    mc = np.abs(s['mean_coef'])
    order = np.argsort(mc.ravel())[::-1]
    rank_of = {(int(fl // N_HEADS), int(fl % N_HEADS)): r + 1
               for r, fl in enumerate(order)}
    print(f"\n  Near-miss ranks in the {cv_label} CV (bind_avg):")
    print(f"    {'head':>8s}  {'folds/5':>7s}  {'mean|coef|':>10s}  {'rank by |coef|':>14s}"
          f"   (of {N_LAYERS*N_HEADS} heads)")
    for (l, h) in sorted(heads):
        nf = counter.get((l, h), 0)
        coef = float(mc[l, h])
        rk = rank_of[(l, h)]
        out[(l, h)] = {'folds': nf, 'mean_abs_coef': coef, 'coef_rank': rk}
        print(f"    L{l:02d}H{h:02d}  {nf:>7d}  {coef:10.4f}  {rk:>14d}")
    return out


# S-heads under the K discovery (were the binding heads "almost selected" on K?)
near_S_in_K = near_miss_table(cv_K, S_HEADS_LIST, "K-task")
# K-heads under the S discovery (vice versa)
near_K_in_S = near_miss_table(cv_S, K_HEADS_LIST, "S-task")

# Layer-distribution comparison
k_layers = sorted(l for l, h in K_HEADS_LIST)
s_layers = sorted(l for l, h in S_HEADS_LIST)
print(f"\n  Layer distribution:")
print(f"    S-head layers: {s_layers}  (published S-heads sit in the L7-L16 band across models)")
print(f"    K-head layers: {k_layers if k_layers else '(none)'}")
if K_HEADS_LIST:
    in_band = [(l, h) for (l, h) in K_HEADS_LIST if 7 <= l <= 16]
    print(f"    K-heads inside L7-L16: {len(in_band)}/{len(K_HEADS_LIST)}  "
          f"({fmt_heads(in_band) if in_band else '{}'})")
    print(f"    K-heads outside L7-L16: {fmt_heads([x for x in K_HEADS_LIST if x not in in_band]) if len(in_band) < len(K_HEADS_LIST) else '{}'}")

print(f"\n[cell 7 done in {time.time()-_t_cell:.1f}s]")


## Cell 8 — Causal cross-test (BASE, K-ONLY): group knockout of the K-heads on the knowledge task (+ GATE: mechanism sanity on 5 prompts; Δ(K) cross-check vs cell 4)

Fill `K_HEADS_PRIMARY` (and optionally `K_HEADS_VARIANT`) for the active model after the cell-6 discovery (+ cell-8b per-head gate). S-side passes intentionally dropped (base |ΔS| = 0.04–0.19 makes S percentages unusable). Reductions are reported in % **and** in absolute Δ(K) units.

The per-model `K_HEADS_PRIMARY` / `K_HEADS_VARIANT` rows below are pre-filled.


In [ ]:
# ================================================================
# CELL 8 — CAUSAL CROSS-TEST (BASE, K-ONLY). Primary + optional variant.
# Baseline K computed once; only the KO passes repeat per subset.
# S-side passes intentionally dropped (K-only decision).
# ================================================================
_t_cell = time.time()

# Subsets to knock out as a group (per model). Fill after the cell-6
# discovery + cell-8b per-head results. The variant subset may be None.
K_HEADS_PRIMARY = {
    "mistral": [(8, 16), (9, 23)],
    "gemma": [(11, 14), (13, 13)],
    "llama": [(7, 7), (8, 17)],
    "nemo": [(8, 9), (10, 24)],
}
K_HEADS_VARIANT = {
    "mistral": [(8, 16), (9, 23), (12, 9)],
    "gemma": [(11, 14), (13, 13), (16,14), (16,15)],
    "llama": [(7, 7), (8, 17), (11, 14)],
    "nemo": [(8, 9), (10, 24), (15, 28)],
}

# Published BASE S-head rows on K (tab:ko-dissociation / tab:a-control-knowledge;
# positive = reduction of |dK| under R->item KO, negative = increase):
PUBLISHED_SHEAD_K_BASE = {
    "mistral": {"red_B_K": 42.1, "red_A_K": -4.4},
    "nemo":    {"red_B_K": 19.0, "red_A_K": -3.8},
    "llama":   {"red_B_K": 36.4, "red_A_K": -4.5},
    "gemma":   {"red_B_K": 21.6, "red_A_K": -3.8},
}


def _to_dict(head_list):
    d = {}
    for l, h in head_list:
        d.setdefault(l, []).append(h)
    return d


VARIANTS = [("primary", K_HEADS_PRIMARY.get(ACTIVE_MODEL) or [])]
_var = K_HEADS_VARIANT.get(ACTIVE_MODEL)
if _var:
    VARIANTS.append(("variant", _var))

print("=" * 80)
print(f"CELL 8: CAUSAL CROSS-TEST (K-ONLY, BASE) — {CFG['label']}")
for tag, hl in VARIANTS:
    print(f"  [{tag}] K-heads under test: "
          f"{_to_dict(hl) if hl else '(EMPTY — fill after discovery)'}")
print("=" * 80)

all_ko_results = {}
ko_results = None

if not VARIANTS[0][1]:
    print("\n  !! K_HEADS_PRIMARY is empty for this model — fill it from the cell-6")
    print("  !! discovery (+ cell-8b per-head gate), then re-run this cell.")
else:
    # Positions (knowledge only; base offset-based convention)
    print("\n  Building positions (knowledge)...")
    positions_k = {c: [] for c in k_conditions}
    for c in k_conditions:
        for i in range(n_total):
            pos = find_identity_and_item_positions(
                tokenizer, texts_fmt_k[c][i], knowledge[c][i],
                data['items_cult'][i], data['assoc_pos'][i])
            positions_k[c].append(pos)
        n_none = sum(1 for p in positions_k[c] if p is None)
        print(f"    knowledge {c}: {n_none}/{n_total} pairs without spans "
              f"(KO falls back to no-op)")

    items_arr = np.array(data['items_cult'])

    # Baseline K (once) + Δ(K) cross-check vs cell 4
    reset_eager_attention()
    print(f"\n  [K task] baseline (no KO)...  [{time.time()-_t_cell:.0f}s]")
    k_base = {c: compute_K_scores_base(texts_fmt_k[c], positions_k[c], {}, 'none',
                                       200, f"base/{c}") for c in k_conditions}
    delta_K_base = k_base['K_cult'].mean() - k_base['K_unrel'].mean()
    kdiffs_base = k_base['K_cult'] - k_base['K_unrel']
    _ref = (results_kscore['K_cult']['K'].mean()
            - results_kscore['K_unrel']['K'].mean())
    _dev_pp = max(float(np.abs(k_base['K_cult'] - results_kscore['K_cult']['K']).max()),
                  float(np.abs(k_base['K_unrel'] - results_kscore['K_unrel']['K']).max()))
    _dev_delta = abs(delta_K_base - _ref)
    _tol = max(1e-3, 0.02 * abs(_ref))   # relative 2% with absolute floor
    print(f"    baseline delta(K) = {delta_K_base:.4f}")
    print(f"    cross-check vs cell 4: per-prompt max|dev| = {_dev_pp:.2e} "
          f"(bf16 drift tripwire 1.0),  |Δ(K) dev| = {_dev_delta:.2e} (tol {_tol:.2e})")
    assert _dev_pp < 1.0, "per-prompt deviation too large for bf16 drift — suspect a leak"
    assert _dev_delta < _tol, "Δ(K) baseline disagrees with cell 4 — investigate"

    # Per-variant KO passes
    for tag, k_heads_list in VARIANTS:
        K_DICT = _to_dict(k_heads_list)
        print("\n" + "#" * 80)
        print(f"#  VARIANT [{tag}] — {K_DICT}")
        print("#" * 80)

        KO_CACHE = OUTPUT_DIR / f"{ACTIVE_MODEL}_base_khead_knockout_{tag}.pkl"
        res = None
        if KO_CACHE.exists():
            with open(KO_CACHE, "rb") as f:
                _cand = pickle.load(f)
            if _cand.get('k_heads') == k_heads_list:
                res = _cand
                print(f"  [cache hit] {KO_CACHE.name} — skipping recomputation")
            else:
                print(f"  [cache stale] heads changed — recomputing")

        if res is None:
            # GATE 8.1: KO must change scores for this head set (5 K prompts)
            reset_eager_attention()
            idx5 = [i for i in range(n_total)
                    if positions_k['K_cult'][i] is not None][:5]
            t5 = [texts_fmt_k['K_cult'][i] for i in idx5]
            p5 = [positions_k['K_cult'][i] for i in idx5]
            b5 = compute_K_scores_base(t5, p5, {}, 'none', None)
            k5 = compute_K_scores_base(t5, p5, K_DICT, 'B_to_item', None)
            reset_eager_attention()
            _max_change = float(np.max(np.abs(b5 - k5)))
            assert _max_change > 1e-6, \
                f"GATE 8.1 FAILED [{tag}]: edge knockout changed NO score"
            print(f"  [GATE 8.1 OK] max |change| = {_max_change:.4f} > 0")

            # K task: R->item KO
            reset_eager_attention()
            print(f"  [K task] R->item KO...  [{time.time()-_t_cell:.0f}s]")
            k_ko = {c: compute_K_scores_base(texts_fmt_k[c], positions_k[c],
                                             K_DICT, 'B_to_item', 200,
                                             f"{tag}/ko/{c}")
                    for c in k_conditions}
            delta_K_ko = k_ko['K_cult'].mean() - k_ko['K_unrel'].mean()
            kdiffs_ko = k_ko['K_cult'] - k_ko['K_unrel']
            t_K, p_K = ttest_clustered(kdiffs_base, kdiffs_ko, items_arr)
            red_K = (1 - delta_K_ko / delta_K_base) * 100 \
                if abs(delta_K_base) > 1e-10 else 0.0
            print(f"    KO delta(K) = {delta_K_ko:.4f}  abs change = "
                  f"{delta_K_ko - delta_K_base:+.4f}  reduction = {red_K:+.1f}%  "
                  f"t = {t_K:.3f}, p = {p_K:.6f}")

            # K task: U->item control
            print(f"  [K task] U->item KO (control)...  [{time.time()-_t_cell:.0f}s]")
            k_ctrl = {c: compute_K_scores_base(texts_fmt_k[c], positions_k[c],
                                               K_DICT, 'A_to_item', 200,
                                               f"{tag}/ctrl/{c}")
                      for c in k_conditions}
            delta_K_ctrl = k_ctrl['K_cult'].mean() - k_ctrl['K_unrel'].mean()
            kdiffs_ctrl = k_ctrl['K_cult'] - k_ctrl['K_unrel']
            t_Kc, p_Kc = ttest_clustered(kdiffs_base, kdiffs_ctrl, items_arr)
            red_K_ctrl = (1 - delta_K_ctrl / delta_K_base) * 100 \
                if abs(delta_K_base) > 1e-10 else 0.0
            print(f"    U-ctrl delta(K) = {delta_K_ctrl:.4f}  "
                  f"reduction = {red_K_ctrl:+.1f}%  "
                  f"t = {t_Kc:.3f}, p = {p_Kc:.6f}  (should be small)")
            reset_eager_attention()

            res = {
                'k_heads': k_heads_list, 'tag': tag, 'variant_model': 'base',
                'knowledge': {
                    'delta_baseline': float(delta_K_base),
                    'delta_B_ko': float(delta_K_ko),
                    'delta_A_ko': float(delta_K_ctrl),
                    'reduction_B_pct': float(red_K),
                    'reduction_A_pct': float(red_K_ctrl),
                    't_B': float(t_K), 'p_B': float(p_K),
                    't_A': float(t_Kc), 'p_A': float(p_Kc),
                    'diffs_base': kdiffs_base.tolist(),
                    'diffs_B_ko': kdiffs_ko.tolist(),
                    'diffs_A_ko': kdiffs_ctrl.tolist()},
                'baseline_crosscheck_perprompt_max': float(_dev_pp),
                'baseline_crosscheck_delta_dev': float(_dev_delta),
            }
            with open(KO_CACHE, "wb") as f:
                pickle.dump(res, f)
            print(f"  Saved {KO_CACHE.name}")

        all_ko_results[tag] = res

    ko_results = all_ko_results.get('primary')  # consumed by cell 9

# Printout (one block per variant)
for tag, res in all_ko_results.items():
    k = res['knowledge']
    pub = PUBLISHED_SHEAD_K_BASE[ACTIVE_MODEL]
    print("\n  " + "=" * 64)
    print(f"  KNOCKOUT (K-only, base) [{tag}] — {CFG['label']}  "
          f"({_to_dict(res['k_heads'])})")
    print("  " + "=" * 64)
    print(f"  {'K-heads R->item':30s}  {k['reduction_B_pct']:+7.1f}%   "
          f"absΔ {k['delta_B_ko']-k['delta_baseline']:+.4f}   (p={k['p_B']:.4f})")
    print(f"  {'S-heads R->item (published)':30s}  {pub['red_B_K']:+7.1f}%")
    print(f"  {'K-heads U->item ctrl':30s}  {k['reduction_A_pct']:+7.1f}%   "
          f"absΔ {k['delta_A_ko']-k['delta_baseline']:+.4f}   (p={k['p_A']:.4f})")
    print(f"  {'S-heads U ctrl (published)':30s}  {pub['red_A_K']:+7.1f}%")
    print("  " + "=" * 64)

print(f"\n[cell 8 done in {time.time()-_t_cell:.1f}s]")


## Cell 8b — Per-head knockout, K-ONLY (necessity), standalone & resumable

Edit `HEADS_TO_TEST` with the cell-6 stable set (3+ folds, any of the 3 features), plus the published binding heads if absent. Output: % reduction of |ΔK| **and absolute Δ** (base denominators are tiny: judge on p and absolutes, not on spectacular %).

The per-model `HEADS_TO_TEST` lists are pre-filled in `PER_MODEL` (params cell).


In [ ]:
# ================================================================
# CELL 8b (standalone) — PER-HEAD KO (K-ONLY, BASE), une tête à la fois.
# Autonome: reconstruit positions + baseline K, puis teste chaque tête
# isolément. Reprise via cache (sauvegarde après chaque tête).
# ================================================================
_t_cell = time.time()

# Les têtes à tester, une par une (cell-6 stable set + binding heads)
# (per-model lists defined in PER_MODEL, cell 1)
HEADS_TO_TEST = PER_MODEL[MODEL_KEY]["HEADS_TO_TEST"]


print("=" * 80)
print(f"CELL 8b STANDALONE: PER-HEAD KO (K-ONLY) — {CFG['label']}")
print(f"  {len(HEADS_TO_TEST)} têtes testées une par une")
print("=" * 80)
assert HEADS_TO_TEST, "HEADS_TO_TEST est vide — remplis-le depuis la cell 6"

# Positions (knowledge, base convention)
print("\n  Building positions (knowledge)...")
positions_k = {c: [] for c in k_conditions}
for c in k_conditions:
    for i in range(n_total):
        pos = find_identity_and_item_positions(
            tokenizer, texts_fmt_k[c][i], knowledge[c][i],
            data['items_cult'][i], data['assoc_pos'][i])
        positions_k[c].append(pos)
    n_none = sum(1 for p in positions_k[c] if p is None)
    print(f"    knowledge {c}: {n_none}/{n_total} pairs without spans "
          f"(KO falls back to no-op)")

items_arr = np.array(data['items_cult'])

# Baseline K (sans KO)
reset_eager_attention()
print(f"\n  [K task] baseline (no KO)...  [{time.time()-_t_cell:.0f}s]")
k_base = {c: compute_K_scores_base(texts_fmt_k[c], positions_k[c], {}, 'none',
                                   400, f"baseline/{c}") for c in k_conditions}
delta_K_base = k_base['K_cult'].mean() - k_base['K_unrel'].mean()
kdiffs_base = k_base['K_cult'] - k_base['K_unrel']
print(f"    baseline delta(K) = {delta_K_base:.4f}")
reset_eager_attention()


def _ko_K_only(heads_dict):
    """KO R->item, K task seulement; t-test clusterisé."""
    reset_eager_attention()
    k_ko = {c: compute_K_scores_base(texts_fmt_k[c], positions_k[c],
                                     heads_dict, 'B_to_item', None)
            for c in k_conditions}
    reset_eager_attention()
    d_K = k_ko['K_cult'].mean() - k_ko['K_unrel'].mean()
    kdiffs = k_ko['K_cult'] - k_ko['K_unrel']
    t_K, p_K = ttest_clustered(kdiffs_base, kdiffs, items_arr)
    red_K = (1 - d_K / delta_K_base) * 100 if abs(delta_K_base) > 1e-10 else 0.0
    return {'delta_K': float(d_K), 'abs_change': float(d_K - delta_K_base),
            'red_K_pct': float(red_K), 't_K': float(t_K), 'p_K': float(p_K)}


# Cache résumable
PERHEAD_CACHE = OUTPUT_DIR / f"{ACTIVE_MODEL}_base_perhead_standalone.pkl"
perhead_results = {'single': {}, 'heads': HEADS_TO_TEST,
                   'delta_K_base': float(delta_K_base)}
if PERHEAD_CACHE.exists():
    with open(PERHEAD_CACHE, "rb") as f:
        _c = pickle.load(f)
    if _c.get('heads') == HEADS_TO_TEST:
        perhead_results['single'].update(_c.get('single', {}))
        print(f"  [cache] {len(perhead_results['single'])} têtes déjà calculées (reprise)")
    else:
        print(f"  [cache stale] liste de têtes changée — recalcul complet")


def _save_perhead():
    with open(PERHEAD_CACHE, "wb") as f:
        pickle.dump(perhead_results, f)


# KO d'une seule tête à la fois (K seulement)
print(f"\n  SINGLE-HEAD KO (necessity), R->item, K task only:")
print(f"    {'head':>8s}  {'red dK%':>8s}  {'abs dK':>9s}  {'p(K)':>8s}")
for (l, h) in HEADS_TO_TEST:
    key = f"L{l}H{h}"
    if key not in perhead_results['single']:
        perhead_results['single'][key] = _ko_K_only({l: [h]})
        _save_perhead()  # save après chaque tête -> résumable
    r = perhead_results['single'][key]
    print(f"    {key:>8s}  {r['red_K_pct']:+7.1f}%  {r['abs_change']:+9.4f}  "
          f"{r['p_K']:8.4f}   [{time.time()-_t_cell:.0f}s]")

print(f"\n  Saved {PERHEAD_CACHE}")
print("\n  Lecture (BASE): les dénominateurs |dK| sont petits (0.16-0.96) — juge")
print("  sur p(K) et sur le delta ABSOLU, pas sur des % spectaculaires. Une tête")
print("  passe le gate causal si p survit à la correction multiple (~1e-3 pour")
print("  ~20 têtes) ET si l'effet absolu est non négligeable.")

print(f"\n[cell 8b standalone done in {time.time()-_t_cell:.1f}s]")


## Cell 9 — Summary, decision rule (BASE, K-only), save


In [ ]:
# ================================================================
# CELL 9 — SUMMARY + INTERPRETATION + SAVE (BASE, K-ONLY)
# The base question: does the independent K discovery on the base model
# converge on the (instruct-discovered, instruct->base-transferred)
# binding heads, and does the K-only group KO confirm causality with a
# magnitude comparable to the published base S-head row?
# No S-side KO is run (K-only decision), so the instruct 2x2 double-
# dissociation criterion is not applicable here; the criteria below are
# convergence-oriented.
# ================================================================
_t_cell = time.time()
from datetime import datetime

DECISION_RULE = (
    "BASE K-only rule: (i) K-discovery AUC adequate (> 0.65); (ii) overlap of the "
    "K-stable set with the published binding heads, read with the near-miss table "
    "(high Jaccard or high near-miss fold counts => convergence; low both => "
    "divergence); (iii) group KO of the K-causal subset reduces |dK| significantly, "
    "with magnitude to be compared against the published base S-head row. "
    "No S-side KO: no double-dissociation claim from this notebook alone.")

pub = PUBLISHED_SHEAD_K_BASE[ACTIVE_MODEL]

crit_i = not K_DISCOVERY_WEAK
near_miss_max = max((v['folds'] for v in near_S_in_K.values()), default=0)
converged = bool(K_HEADS_LIST) and (jaccard >= 0.5 or near_miss_max >= 3)
diverged = bool(K_HEADS_LIST) and (jaccard <= 0.2 and near_miss_max < 2)

if ko_results is not None:
    red_KH_K = ko_results['knowledge']['reduction_B_pct']
    p_KH_K = ko_results['knowledge']['p_B']
    ko_line = (f"group KO of {ko_results['k_heads']}: |dK| {red_KH_K:+.1f}% "
               f"(p={p_KH_K:.2e}); published base S-head row: {pub['red_B_K']:+.1f}%")
else:
    red_KH_K = p_KH_K = None
    ko_line = "group KO not run yet (fill K_HEADS_PRIMARY in cell 8)"

if not crit_i:
    outcome = "INCONCLUSIVE (weak discovery)"
    interp = (f"K-head discovery reached mean AUC = {auc_K:.3f} (<= 0.65) on "
              f"{CFG['label']}; the K-stable set is unreliable and no overlap "
              f"conclusion should be drawn from this model.")
elif not K_HEADS_LIST:
    outcome = "INCONCLUSIVE (no stable K-heads)"
    interp = (f"Discovery AUC adequate ({auc_K:.3f}) but no head selected in >= 3/5 "
              f"folds: the base K signal is distributed rather than concentrated. "
              f"Near-miss of binding heads in the K-CV: max {near_miss_max}/5 folds.")
elif converged:
    outcome = "CONVERGENCE on the binding heads"
    interp = (f"The independent base K discovery substantially coincides with the "
              f"published binding heads (Jaccard = {jaccard:.3f}; max binding-head "
              f"fold count in the K-CV = {near_miss_max}/5). {ko_line}. Consistent "
              f"with the pre-training-origin claim: the knowledge pathway of the "
              f"base model runs through the same heads.")
elif diverged:
    outcome = "DIVERGENCE (to be confirmed causally)"
    interp = (f"The base K-stable set differs from the binding heads (Jaccard = "
              f"{jaccard:.3f}; near-miss max {near_miss_max}/5 — not a thresholding "
              f"artifact). {ko_line}. Before claiming a distinct base K circuit, "
              f"run cell 8b on the new candidates: base discovery can select "
              f"causally inert heads (as the instruct per-head scan showed).")
else:
    outcome = "MIXED / PARTIAL"
    interp = (f"Partial overlap (Jaccard = {jaccard:.3f}; near-miss max "
              f"{near_miss_max}/5). {ko_line}. Report the numbers without a "
              f"categorical claim; the per-head gate (cell 8b) arbitrates which "
              f"candidates are causally real.")

print("=" * 80)
print(f"CELL 9: SUMMARY — {CFG['label']} (BASE, K-only)")
print("=" * 80)
print(f"\n  DECISION RULE: {DECISION_RULE}")
print(f"\n  K discovery:  AUC = {auc_K:.3f}  (weak = {K_DISCOVERY_WEAK})")
print(f"  K-stable set: {[f'L{l}H{h}' for l, h in K_HEADS_LIST]}")
print(f"  Jaccard vs published binding heads: {jaccard:.3f}  "
      f"(near-miss max {near_miss_max}/5)")
print(f"  KO: {ko_line}")
print(f"\n  OUTCOME: {outcome}")
print(f"\n  {interp}")

summary = {
    'meta': {'model': ACTIVE_MODEL, 'variant': 'base', 'label': CFG['label'],
             'timestamp': datetime.now().isoformat(), 'k_only': True},
    'decision_rule': DECISION_RULE,
    'auc_K': float(auc_K), 'K_DISCOVERY_WEAK': bool(K_DISCOVERY_WEAK),
    'K_HEADS_LIST': K_HEADS_LIST, 'jaccard': float(jaccard),
    'near_S_in_K': {f"L{l}H{h}": v for (l, h), v in near_S_in_K.items()},
    'ko_results': all_ko_results,
    'outcome': outcome, 'interpretation': interp,
}
with open(OUTPUT_DIR / f"{ACTIVE_MODEL}_base_khead_summary.pkl", "wb") as f:
    pickle.dump(summary, f)
print(f"\n  Saved {OUTPUT_DIR / (ACTIVE_MODEL + '_base_khead_summary.pkl')}")
print(f"\n[cell 9 done in {time.time()-_t_cell:.1f}s]")
